In [2]:
%pip install scipy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Statistical Analysis
### Statistical Testing of Numerical Variables

In [3]:
from scipy.stats import ttest_ind

In [4]:
import pandas as pd

df = pd.read_csv("../data/processed/customer_churn_clean.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
churned = df[df["Churn"] == "Yes"]
retained = df[df["Churn"] == "No"]

In [6]:
numerical_variables = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

statistical_results = []

for variable in numerical_variables:
    
    churned_values = churned[variable].dropna()
    retained_values = retained[variable].dropna()
    
    t_stat, p_value = ttest_ind(
        churned_values,
        retained_values,
        equal_var=False
    )
    
    statistical_results.append({
        "Variable": variable,
        "Churned Mean": churned_values.mean(),
        "Retained Mean": retained_values.mean(),
        "Mean Difference": (
            churned_values.mean()
            - retained_values.mean()
        ),
        "t-statistic": t_stat,
        "p-value": p_value
    })

t_test_results = pd.DataFrame(statistical_results)

t_test_results

alpha = 0.05

t_test_results["Statistically Significant"] = (
    t_test_results["p-value"] < alpha
)

t_test_results

,Variable,Churned Mean,Retained Mean,Mean Difference,t-statistic,p-value,Statistically Significant
0,tenure,17.979133,37.569965,-19.590832,-34.823819,1.195495e-232,True
1,MonthlyCharges,74.441332,61.265124,13.176209,18.407527,8.592449e-73,True
2,TotalCharges,1531.796094,2549.911442,-1018.115348,-18.706618,5.902581e-75,True


In [7]:
t_test_display = t_test_results.copy()

t_test_display["Churned Mean"] = (
    t_test_display["Churned Mean"].round(2)
)

t_test_display["Retained Mean"] = (
    t_test_display["Retained Mean"].round(2)
)

t_test_display["Mean Difference"] = (
    t_test_display["Mean Difference"].round(2)
)

t_test_display["t-statistic"] = (
    t_test_display["t-statistic"].round(3)
)

t_test_display["p-value"] = (
    t_test_display["p-value"].apply(
        lambda x: f"{x:.4e}"
    )
)

t_test_display

,Variable,Churned Mean,Retained Mean,Mean Difference,t-statistic,p-value,Statistically Significant
0,tenure,17.98,37.57,-19.59,-34.824,1.1955e-232,True
1,MonthlyCharges,74.44,61.27,13.18,18.408,8.5924e-73,True
2,TotalCharges,1531.80,2549.91,-1018.12,-18.707,5.9026e-75,True


In [8]:
for _, row in t_test_results.iterrows():
    
    variable = row["Variable"]
    p_value = row["p-value"]
    
    if p_value < alpha:
        conclusion = "Statistically significant difference"
    else:
        conclusion = "No statistically significant difference"
    
    print(
        f"{variable}: {conclusion} "
        f"(p-value = {p_value:.4e})"
    )

tenure: Statistically significant difference (p-value = 1.1955e-232)
MonthlyCharges: Statistically significant difference (p-value = 8.5924e-73)
TotalCharges: Statistically significant difference (p-value = 5.9026e-75)


In [9]:
for _, row in t_test_results.iterrows():
    
    variable = row["Variable"]
    
    if row["Churned Mean"] > row["Retained Mean"]:
        direction = "higher among churned customers"
    else:
        direction = "lower among churned customers"
    
    print(
        f"{variable}: {direction}"
    )

tenure: lower among churned customers
MonthlyCharges: higher among churned customers
TotalCharges: lower among churned customers


### Business Interpretation

Welch's independent two-sample t-tests were used to determine whether the mean values of key numerical variables differed significantly between churned and retained customers.

The results indicate statistically significant differences in key customer characteristics such as tenure, MonthlyCharges, and TotalCharges.

Churned customers tend to have substantially lower tenure than retained customers, supporting the earlier EDA finding that newer customers represent a higher-risk lifecycle segment.

Churned customers also tend to have higher MonthlyCharges, suggesting that customers with higher recurring charges may represent an important retention segment.

TotalCharges shows a statistically significant difference between the two groups, but this result should be interpreted alongside tenure because cumulative charges are strongly related to how long a customer has remained with the company.

Statistical significance indicates that the observed differences are unlikely to be explained solely by random sampling variation. However, statistical significance does not establish causation or determine the practical importance of a variable by itself.

###  Categorical Variables vs Churn




In [10]:
categorical_variables = [
    "Contract",
    "PaymentMethod",
    "InternetService",
    "OnlineSecurity",
    "TechSupport",
    "PaperlessBilling",
    "Partner",
    "Dependents",
    "SeniorCitizen"
]

categorical_variables

['Contract',
 'PaymentMethod',
 'InternetService',
 'OnlineSecurity',
 'TechSupport',
 'PaperlessBilling',
 'Partner',
 'Dependents',
 'SeniorCitizen']

In [11]:
from scipy.stats import chi2_contingency

In [12]:
contract_churn_table = pd.crosstab(
    df["Contract"],
    df["Churn"]
)

contract_churn_table

Churn,No,Yes
Contract,,
Month-to-month,2220,1655
One year,1307,166
Two year,1647,48


In [13]:
chi2_stat, p_value, degrees_of_freedom, expected_frequencies = chi2_contingency(
    contract_churn_table
)

print("Chi-Square Statistic:", chi2_stat)
print("p-value:", p_value)
print("Degrees of Freedom:", degrees_of_freedom)
print("Expected Frequencies:", expected_frequencies)

Chi-Square Statistic: 1184.5965720837926
p-value: 5.863038300673391e-258
Degrees of Freedom: 2
Expected Frequencies: [[2846.69175067 1028.30824933]
 [1082.11018032  390.88981968]
 [1245.198069    449.801931  ]]


In [14]:
chi_square_results = []

for variable in categorical_variables:

    contingency_table = pd.crosstab(
        df[variable],
        df["Churn"]
    )

    chi2_stat, p_value, degrees_of_freedom, expected_frequencies = chi2_contingency(
        contingency_table
    )

    chi_square_results.append({
        "Variable": variable,
        "Chi-Square Statistic": chi2_stat,
        "Degrees of Freedom": degrees_of_freedom,
        "p-value": p_value
    })

chi_square_results = pd.DataFrame(chi_square_results)

chi_square_results

,Variable,Chi-Square Statistic,Degrees of Freedom,p-value
0,Contract,1184.596572,2,5.863038e-258
1,PaymentMethod,648.142327,3,3.682355e-140
2,InternetService,732.309590,2,9.571788e-160
3,OnlineSecurity,849.998968,2,2.661150e-185
4,TechSupport,828.197068,2,1.443084e-180
5,PaperlessBilling,258.277649,1,4.073355e-58
6,Partner,158.733382,1,2.139911e-36
7,Dependents,189.129249,1,4.924922e-43
8,SeniorCitizen,159.426300,1,1.510067e-36


In [15]:
alpha = 0.05

chi_square_results["Statistically Significant"] = (
    chi_square_results["p-value"] < alpha
)

chi_square_results

chi_square_results = chi_square_results.sort_values(
    by="p-value"
).reset_index(drop=True)

chi_square_results

,Variable,Chi-Square Statistic,Degrees of Freedom,p-value,Statistically Significant
0,Contract,1184.596572,2,5.863038e-258,True
1,OnlineSecurity,849.998968,2,2.661150e-185,True
2,TechSupport,828.197068,2,1.443084e-180,True
3,InternetService,732.309590,2,9.571788e-160,True
4,PaymentMethod,648.142327,3,3.682355e-140,True
5,PaperlessBilling,258.277649,1,4.073355e-58,True
6,Dependents,189.129249,1,4.924922e-43,True
7,SeniorCitizen,159.426300,1,1.510067e-36,True
8,Partner,158.733382,1,2.139911e-36,True


In [16]:
categorical_test_summary = chi_square_results[
    [
        "Variable",
        "Chi-Square Statistic",
        "Degrees of Freedom",
        "p-value",
        "Statistically Significant"
    ]
].copy()

categorical_test_summary

,Variable,Chi-Square Statistic,Degrees of Freedom,p-value,Statistically Significant
0,Contract,1184.596572,2,5.863038e-258,True
1,OnlineSecurity,849.998968,2,2.661150e-185,True
2,TechSupport,828.197068,2,1.443084e-180,True
3,InternetService,732.309590,2,9.571788e-160,True
4,PaymentMethod,648.142327,3,3.682355e-140,True
5,PaperlessBilling,258.277649,1,4.073355e-58,True
6,Dependents,189.129249,1,4.924922e-43,True
7,SeniorCitizen,159.426300,1,1.510067e-36,True
8,Partner,158.733382,1,2.139911e-36,True


In [17]:
significant_categorical_variables = chi_square_results[
    chi_square_results["Statistically Significant"]
]["Variable"].tolist()

significant_categorical_variables

['Contract',
 'OnlineSecurity',
 'TechSupport',
 'InternetService',
 'PaymentMethod',
 'PaperlessBilling',
 'Dependents',
 'SeniorCitizen',
 'Partner']

In [18]:
for variable in significant_categorical_variables:

    churn_rate = (
        df.groupby(variable, observed=False)["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
        .sort_values(ascending=False)
    )

    print(f"\n{variable}")
    print(churn_rate)


Contract
Contract
Month-to-month    42.709677
One year          11.269518
Two year           2.831858
Name: Churn, dtype: float64

OnlineSecurity
OnlineSecurity
No                     41.766724
Yes                    14.611194
No internet service     7.404980
Name: Churn, dtype: float64

TechSupport
TechSupport
No                     41.635474
Yes                    15.166341
No internet service     7.404980
Name: Churn, dtype: float64

InternetService
InternetService
Fiber optic    41.892765
DSL            18.959108
No              7.404980
Name: Churn, dtype: float64

PaymentMethod
PaymentMethod
Electronic check             45.285412
Mailed check                 19.106700
Bank transfer (automatic)    16.709845
Credit card (automatic)      15.243101
Name: Churn, dtype: float64

PaperlessBilling
PaperlessBilling
Yes    33.565092
No     16.330084
Name: Churn, dtype: float64

Dependents
Dependents
No     31.279140
Yes    15.450237
Name: Churn, dtype: float64

SeniorCitizen
SeniorCitizen

##  Chi-Square Statistical Analysis

### Hypotheses
H₀: The variable and churn are independent.

H₁: The variable and churn are associated.

### Significance Level
Alpha = 0.05.

A p-value below 0.05 indicates a statistically significant association.

### Key Insight
Several customer characteristics showed statistically significant
associations with churn.

Contract, payment method, internet service, online security and
technical support are particularly relevant for further analysis.

### Business Relevance
The results help identify customer characteristics that may be useful
for churn segmentation and machine-learning models.



### Effect Size Analysis

### Cohen's d for Numerical Variables

In [19]:
import numpy as np
def cohens_d(group1, group2):
    n1 = len(group1)
    n2 = len(group2)

    mean1 = group1.mean()
    mean2 = group2.mean()

    std1 = group1.std()
    std2 = group2.std()

    pooled_std = np.sqrt(
        ((n1 - 1) * std1**2 + (n2 - 1) * std2**2)
        / (n1 + n2 - 2)
    )

    d = (mean1 - mean2) / pooled_std

    return d

In [20]:
numerical_variables = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

effect_size_results = []

for variable in numerical_variables:

    churned_values = df[df["Churn"] == "Yes"][variable].dropna()
    retained_values = df[df["Churn"] == "No"][variable].dropna()

    d = cohens_d(churned_values, retained_values)

    effect_size_results.append({
        "Variable": variable,
        "Cohens_d": d
    })

numerical_effect_sizes = pd.DataFrame(effect_size_results)

numerical_effect_sizes

,Variable,Cohens_d
0,tenure,-0.852250
1,MonthlyCharges,0.446283
2,TotalCharges,-0.458213


In [21]:
def interpret_cohens_d(d):
    absolute_d = abs(d)

    if absolute_d < 0.2:
        return "Negligible"
    elif absolute_d < 0.5:
        return "Small"
    elif absolute_d < 0.8:
        return "Medium"
    else:
        return "Large"


numerical_effect_sizes["Effect Size Interpretation"] = (
    numerical_effect_sizes["Cohens_d"].apply(interpret_cohens_d)
)

numerical_effect_sizes

,Variable,Cohens_d,Effect Size Interpretation
0,tenure,-0.852250,Large
1,MonthlyCharges,0.446283,Small
2,TotalCharges,-0.458213,Small


In [22]:
numerical_effect_analysis = t_test_results.merge(
    numerical_effect_sizes,
    on="Variable",
    how="left"
)

numerical_effect_analysis = numerical_effect_analysis.sort_values(
    by="Cohens_d",
    key=lambda x: x.abs(),
    ascending=False
).reset_index(drop=True)

numerical_effect_analysis

,Variable,Churned Mean,Retained Mean,Mean Difference,t-statistic,p-value,Statistically Significant,Cohens_d,Effect Size Interpretation
0,tenure,17.979133,37.569965,-19.590832,-34.823819,1.195495e-232,True,-0.852250,Large
1,TotalCharges,1531.796094,2549.911442,-1018.115348,-18.706618,5.902581e-75,True,-0.458213,Small
2,MonthlyCharges,74.441332,61.265124,13.176209,18.407527,8.592449e-73,True,0.446283,Small


### Cramér's V for Categorical Variables

In [23]:
def cramers_v(contingency_table):

    chi2 = chi2_contingency(contingency_table)[0]

    n = contingency_table.sum().sum()

    rows, columns = contingency_table.shape

    phi2 = chi2 / n

    phi2_corrected = max(
        0,
        phi2 - ((columns - 1) * (rows - 1)) / (n - 1)
    )

    rows_corrected = rows - (
        ((rows - 1) ** 2) / (n - 1)
    )

    columns_corrected = columns - (
        ((columns - 1) ** 2) / (n - 1)
    )

    denominator = min(
        columns_corrected - 1,
        rows_corrected - 1
    )

    return np.sqrt(phi2_corrected / denominator)

In [24]:
categorical_effect_sizes = []

for variable in categorical_variables:

    contingency_table = pd.crosstab(
        df[variable],
        df["Churn"]
    )

    v = cramers_v(contingency_table)

    categorical_effect_sizes.append({
        "Variable": variable,
        "Cramers_V": v
    })

categorical_effect_sizes = pd.DataFrame(
    categorical_effect_sizes
)

categorical_effect_sizes

,Variable,Cramers_V
0,Contract,0.409798
1,PaymentMethod,0.302677
2,InternetService,0.322037
3,OnlineSecurity,0.347016
4,TechSupport,0.342526
5,PaperlessBilling,0.191141
6,Partner,0.149663
7,Dependents,0.163448
8,SeniorCitizen,0.149991


In [25]:
def interpret_cramers_v(v):

    if v < 0.1:
        return "Very weak"
    elif v < 0.2:
        return "Weak"
    elif v < 0.3:
        return "Moderate"
    else:
        return "Strong"


categorical_effect_sizes["Association Strength"] = (
    categorical_effect_sizes["Cramers_V"].apply(
        interpret_cramers_v
    )
)

categorical_effect_sizes

,Variable,Cramers_V,Association Strength
0,Contract,0.409798,Strong
1,PaymentMethod,0.302677,Strong
2,InternetService,0.322037,Strong
3,OnlineSecurity,0.347016,Strong
4,TechSupport,0.342526,Strong
5,PaperlessBilling,0.191141,Weak
6,Partner,0.149663,Weak
7,Dependents,0.163448,Weak
8,SeniorCitizen,0.149991,Weak


In [26]:
categorical_effect_analysis = chi_square_results.merge(
    categorical_effect_sizes,
    on="Variable",
    how="left"
)

categorical_effect_analysis = categorical_effect_analysis.sort_values(
    by="Cramers_V",
    ascending=False
).reset_index(drop=True)

categorical_effect_analysis

,Variable,Chi-Square Statistic,Degrees of Freedom,p-value,Statistically Significant,Cramers_V,Association Strength
0,Contract,1184.596572,2,5.863038e-258,True,0.409798,Strong
1,OnlineSecurity,849.998968,2,2.661150e-185,True,0.347016,Strong
2,TechSupport,828.197068,2,1.443084e-180,True,0.342526,Strong
3,InternetService,732.309590,2,9.571788e-160,True,0.322037,Strong
4,PaymentMethod,648.142327,3,3.682355e-140,True,0.302677,Strong
5,PaperlessBilling,258.277649,1,4.073355e-58,True,0.191141,Weak
6,Dependents,189.129249,1,4.924922e-43,True,0.163448,Weak
7,SeniorCitizen,159.426300,1,1.510067e-36,True,0.149991,Weak
8,Partner,158.733382,1,2.139911e-36,True,0.149663,Weak


In [27]:
print("NUMERICAL VARIABLES — EFFECT SIZE")
display(
    numerical_effect_analysis[
        [
            "Variable",
            "Churned Mean",
            "Retained Mean",
            "Mean Difference",
            "p-value",
            "Cohens_d",
            "Effect Size Interpretation"
        ]
    ]
)

print("\nCATEGORICAL VARIABLES — EFFECT SIZE")
display(
    categorical_effect_analysis[
        [
            "Variable",
            "p-value",
            "Statistically Significant",
            "Cramers_V",
            "Association Strength"
        ]
    ]
)

NUMERICAL VARIABLES — EFFECT SIZE


,Variable,Churned Mean,Retained Mean,Mean Difference,p-value,Cohens_d,Effect Size Interpretation
0,tenure,17.979133,37.569965,-19.590832,1.195495e-232,-0.852250,Large
1,TotalCharges,1531.796094,2549.911442,-1018.115348,5.902581e-75,-0.458213,Small
2,MonthlyCharges,74.441332,61.265124,13.176209,8.592449e-73,0.446283,Small



CATEGORICAL VARIABLES — EFFECT SIZE


,Variable,p-value,Statistically Significant,Cramers_V,Association Strength
0,Contract,5.863038e-258,True,0.409798,Strong
1,OnlineSecurity,2.661150e-185,True,0.347016,Strong
2,TechSupport,1.443084e-180,True,0.342526,Strong
3,InternetService,9.571788e-160,True,0.322037,Strong
4,PaymentMethod,3.682355e-140,True,0.302677,Strong
5,PaperlessBilling,4.073355e-58,True,0.191141,Weak
6,Dependents,4.924922e-43,True,0.163448,Weak
7,SeniorCitizen,1.510067e-36,True,0.149991,Weak
8,Partner,2.139911e-36,True,0.149663,Weak


##  Effect Size Analysis


### Methods
- Cohen's d → numerical variables
- Cramér's V → categorical variables

### Key Insight
Statistical significance shows whether evidence of a relationship exists,
while effect size indicates how strong that relationship is.

### Business Relevance
Effect sizes help prioritize variables beyond simply ranking them by
p-value and support better feature selection for later modeling.


## Multiple Testing Correction

In [28]:
# Creating a combined table of all hypothesis tests

numerical_pvalues = t_test_results[
    ["Variable", "p-value"]
].copy()

numerical_pvalues["Test Type"] = "Welch's t-test"

categorical_pvalues = chi_square_results[
    ["Variable", "p-value"]
].copy()

categorical_pvalues["Test Type"] = "Chi-Square"

all_statistical_tests = pd.concat(
    [
        numerical_pvalues,
        categorical_pvalues
    ],
    ignore_index=True
)

all_statistical_tests

,Variable,p-value,Test Type
0,tenure,1.195495e-232,Welch's t-test
1,MonthlyCharges,8.592449e-73,Welch's t-test
2,TotalCharges,5.902581e-75,Welch's t-test
3,Contract,5.863038e-258,Chi-Square
4,OnlineSecurity,2.661150e-185,Chi-Square
5,TechSupport,1.443084e-180,Chi-Square
6,InternetService,9.571788e-160,Chi-Square
7,PaymentMethod,3.682355e-140,Chi-Square
8,PaperlessBilling,4.073355e-58,Chi-Square
9,Dependents,4.924922e-43,Chi-Square


In [29]:
%pip install statsmodels


   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 1.7 MB/s eta 0:00:07
   --- ------------------------------------ 1.0/11.3 MB 1.9 MB/s eta 0:00:06
   ------ --------------------------------- 1.8/11.3 MB 2.7 MB/s eta 0:00:04
   ----------- ---------------------------- 3.1/11.3 MB 3.5 MB/s eta 0:00:03
   ----------------- ---------------------- 5.0/11.3 MB 4.6 MB/s eta 0:00:02
   ------------------------- -------------- 7.3/11.3 MB 5.6 MB/s eta 0:00:01
   -------------------------------- ------- 9.2/11.3 MB 6.2 MB/s eta 0:00:01
   ------------------------------------- -- 10.7/11.3 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 6.2 MB/s  0:00:01

   ---------------------------------------- 0/6 [wrapt]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
from statsmodels.stats.multitest import multipletests

rejection, adjusted_pvalues, _, _ = multipletests(
    all_statistical_tests["p-value"],
    alpha=0.05,
    method="fdr_bh"
)

all_statistical_tests["Adjusted p-value"] = adjusted_pvalues

all_statistical_tests["Significant After FDR"] = rejection

all_statistical_tests

,Variable,p-value,Test Type,Adjusted p-value,Significant After FDR
0,tenure,1.195495e-232,Welch's t-test,7.172967e-232,True
1,MonthlyCharges,8.592449e-73,Welch's t-test,1.288867e-72,True
2,TotalCharges,5.902581e-75,Welch's t-test,1.011871e-74,True
3,Contract,5.863038e-258,Chi-Square,7.035646e-257,True
4,OnlineSecurity,2.661150e-185,Chi-Square,1.064460e-184,True
5,TechSupport,1.443084e-180,Chi-Square,4.329252e-180,True
6,InternetService,9.571788e-160,Chi-Square,2.297229e-159,True
7,PaymentMethod,3.682355e-140,Chi-Square,7.364709e-140,True
8,PaperlessBilling,4.073355e-58,Chi-Square,5.431140e-58,True
9,Dependents,4.924922e-43,Chi-Square,5.909906e-43,True


In [31]:
all_statistical_tests["Original Significant"] = (
    all_statistical_tests["p-value"] < 0.05
)

comparison = all_statistical_tests[
    [
        "Variable",
        "Test Type",
        "p-value",
        "Adjusted p-value",
        "Original Significant",
        "Significant After FDR"
    ]
].sort_values(
    by="Adjusted p-value"
)

display(comparison)

,Variable,Test Type,p-value,Adjusted p-value,Original Significant,Significant After FDR
3,Contract,Chi-Square,5.863038e-258,7.035646e-257,True,True
0,tenure,Welch's t-test,1.195495e-232,7.172967e-232,True,True
4,OnlineSecurity,Chi-Square,2.661150e-185,1.064460e-184,True,True
5,TechSupport,Chi-Square,1.443084e-180,4.329252e-180,True,True
6,InternetService,Chi-Square,9.571788e-160,2.297229e-159,True,True
7,PaymentMethod,Chi-Square,3.682355e-140,7.364709e-140,True,True
2,TotalCharges,Welch's t-test,5.902581e-75,1.011871e-74,True,True
1,MonthlyCharges,Welch's t-test,8.592449e-73,1.288867e-72,True,True
8,PaperlessBilling,Chi-Square,4.073355e-58,5.431140e-58,True,True
9,Dependents,Chi-Square,4.924922e-43,5.909906e-43,True,True


In [32]:
fdr_significant_variables = all_statistical_tests[
    all_statistical_tests["Significant After FDR"]
]["Variable"].tolist()

print("Variables remaining significant after FDR correction:")
print(fdr_significant_variables)

Variables remaining significant after FDR correction:
['tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'OnlineSecurity', 'TechSupport', 'InternetService', 'PaymentMethod', 'PaperlessBilling', 'Dependents', 'SeniorCitizen', 'Partner']


##  Multiple Testing Correction

### Why It Was Used
Performing multiple tests increases the chance of false-positive results.

FDR correction controls the expected proportion of false discoveries among
the results identified as significant.

### Analysis
The p-values from 3 numerical and 9 categorical tests were combined and
adjusted using the Benjamini-Hochberg method.

### Output
Original p-values were compared with FDR-adjusted p-values to identify
relationships that remained statistically significant.

### Business Relevance
Variables remaining significant after correction provide more robust
statistical evidence for further investigation and modeling.


## Statistical Relationship & Business Priority Matrix

In [33]:
numerical_priority = numerical_effect_analysis[
    [
        "Variable",
        "Churned Mean",
        "Retained Mean",
        "Mean Difference",
        "p-value",
        "Cohens_d",
        "Effect Size Interpretation"
    ]
].copy()

numerical_priority["Absolute Effect Size"] = (
    numerical_priority["Cohens_d"].abs()
)

numerical_priority = numerical_priority.sort_values(
    by="Absolute Effect Size",
    ascending=False
).reset_index(drop=True)

display(numerical_priority)

,Variable,Churned Mean,Retained Mean,Mean Difference,p-value,Cohens_d,Effect Size Interpretation,Absolute Effect Size
0,tenure,17.979133,37.569965,-19.590832,1.195495e-232,-0.852250,Large,0.852250
1,TotalCharges,1531.796094,2549.911442,-1018.115348,5.902581e-75,-0.458213,Small,0.458213
2,MonthlyCharges,74.441332,61.265124,13.176209,8.592449e-73,0.446283,Small,0.446283


In [34]:
categorical_priority = categorical_effect_analysis[
    [
        "Variable",
        "p-value",
        "Statistically Significant",
        "Cramers_V",
        "Association Strength"
    ]
].copy()

categorical_priority = categorical_priority.sort_values(
    by="Cramers_V",
    ascending=False
).reset_index(drop=True)

display(categorical_priority)

,Variable,p-value,Statistically Significant,Cramers_V,Association Strength
0,Contract,5.863038e-258,True,0.409798,Strong
1,OnlineSecurity,2.661150e-185,True,0.347016,Strong
2,TechSupport,1.443084e-180,True,0.342526,Strong
3,InternetService,9.571788e-160,True,0.322037,Strong
4,PaymentMethod,3.682355e-140,True,0.302677,Strong
5,PaperlessBilling,4.073355e-58,True,0.191141,Weak
6,Dependents,4.924922e-43,True,0.163448,Weak
7,SeniorCitizen,1.510067e-36,True,0.149991,Weak
8,Partner,2.139911e-36,True,0.149663,Weak


In [35]:
categorical_business_summary = []

for variable in categorical_variables:

    churn_rates = (
        df.groupby(variable, observed=False)["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
    )

    highest_risk_category = churn_rates.idxmax()
    highest_churn_rate = churn_rates.max()

    categorical_business_summary.append({
        "Variable": variable,
        "Highest Risk Category": highest_risk_category,
        "Highest Churn Rate (%)": highest_churn_rate
    })

categorical_business_summary = pd.DataFrame(
    categorical_business_summary
)

categorical_business_summary

,Variable,Highest Risk Category,Highest Churn Rate (%)
0,Contract,Month-to-month,42.709677
1,PaymentMethod,Electronic check,45.285412
2,InternetService,Fiber optic,41.892765
3,OnlineSecurity,No,41.766724
4,TechSupport,No,41.635474
5,PaperlessBilling,Yes,33.565092
6,Partner,No,32.957979
7,Dependents,No,31.279140
8,SeniorCitizen,1,41.681261


In [36]:
categorical_priority_final = categorical_priority.merge(
    categorical_business_summary,
    on="Variable",
    how="left"
)

categorical_priority_final = categorical_priority_final.sort_values(
    by="Cramers_V",
    ascending=False
).reset_index(drop=True)

display(categorical_priority_final)

,Variable,p-value,Statistically Significant,Cramers_V,Association Strength,Highest Risk Category,Highest Churn Rate (%)
0,Contract,5.863038e-258,True,0.409798,Strong,Month-to-month,42.709677
1,OnlineSecurity,2.661150e-185,True,0.347016,Strong,No,41.766724
2,TechSupport,1.443084e-180,True,0.342526,Strong,No,41.635474
3,InternetService,9.571788e-160,True,0.322037,Strong,Fiber optic,41.892765
4,PaymentMethod,3.682355e-140,True,0.302677,Strong,Electronic check,45.285412
5,PaperlessBilling,4.073355e-58,True,0.191141,Weak,Yes,33.565092
6,Dependents,4.924922e-43,True,0.163448,Weak,No,31.279140
7,SeniorCitizen,1.510067e-36,True,0.149991,Weak,1,41.681261
8,Partner,2.139911e-36,True,0.149663,Weak,No,32.957979


In [37]:
top_categorical_drivers = categorical_priority_final.head(5)

top_numerical_drivers = numerical_priority.head(3)

print("TOP CATEGORICAL DRIVERS")
display(top_categorical_drivers)

print("\nTOP NUMERICAL DRIVERS")
display(top_numerical_drivers)

TOP CATEGORICAL DRIVERS


,Variable,p-value,Statistically Significant,Cramers_V,Association Strength,Highest Risk Category,Highest Churn Rate (%)
0,Contract,5.863038e-258,True,0.409798,Strong,Month-to-month,42.709677
1,OnlineSecurity,2.661150e-185,True,0.347016,Strong,No,41.766724
2,TechSupport,1.443084e-180,True,0.342526,Strong,No,41.635474
3,InternetService,9.571788e-160,True,0.322037,Strong,Fiber optic,41.892765
4,PaymentMethod,3.682355e-140,True,0.302677,Strong,Electronic check,45.285412



TOP NUMERICAL DRIVERS


,Variable,Churned Mean,Retained Mean,Mean Difference,p-value,Cohens_d,Effect Size Interpretation,Absolute Effect Size
0,tenure,17.979133,37.569965,-19.590832,1.195495e-232,-0.852250,Large,0.852250
1,TotalCharges,1531.796094,2549.911442,-1018.115348,5.902581e-75,-0.458213,Small,0.458213
2,MonthlyCharges,74.441332,61.265124,13.176209,8.592449e-73,0.446283,Small,0.446283


In [38]:
statistical_summary = pd.DataFrame({
    "Analysis": [
        "Numerical hypothesis testing",
        "Categorical hypothesis testing",
        "Numerical effect size",
        "Categorical effect size",
        "Multiple testing correction"
    ],
    "Method": [
        "Welch's t-test",
        "Chi-Square Test",
        "Cohen's d",
        "Cramér's V",
        "Benjamini-Hochberg FDR"
    ],
    "Purpose": [
        "Identify differences between churned and retained customers",
        "Identify categorical variables associated with churn",
        "Measure strength of numerical differences",
        "Measure strength of categorical associations",
        "Improve robustness of statistical conclusions"
    ]
})

display(statistical_summary)

,Analysis,Method,Purpose
0,Numerical hypothesis testing,Welch's t-test,Identify differences between churned and retai...
1,Categorical hypothesis testing,Chi-Square Test,Identify categorical variables associated with...
2,Numerical effect size,Cohen's d,Measure strength of numerical differences
3,Categorical effect size,Cramér's V,Measure strength of categorical associations
4,Multiple testing correction,Benjamini-Hochberg FDR,Improve robustness of statistical conclusions


##  Statistical Business Priority

### Objective
Consolidate statistical findings into a business-oriented priority view.

### Analysis
Numerical variables were ranked using absolute Cohen's d, while categorical
variables were ranked using Cramér's V.

Category-level churn rates were also calculated to provide business context.

### Key Insight
Statistical significance, effect size, churn rate, and business impact
should be considered together rather than relying only on p-values.

### Business Relevance
The analysis identifies strong churn-risk indicators that can be explored
further during feature engineering and machine-learning modeling.

### Conclusion
The statistical analysis established:
- Numerical differences using Welch's t-test
- Categorical associations using Chi-Square
- Effect sizes using Cohen's d and Cramér's V
- Robustness using FDR correction
- Business priority using combined statistical evidence